In [16]:
import pandas as pd 
from glob import glob 
root_dir = '/home/work/yuna/HPA/evaluation/scored'

model_names = [
            "OpenGVLab/InternVL3_5-8B",
            "OpenGVLab/InternVL3_5-2B",
            "OpenGVLab/InternVL3_5-4B",
            "OpenGVLab/InternVL3_5-1B",

            "Qwen/Qwen3-VL-2B-Instruct", 
            "Qwen/Qwen3-VL-4B-Instruct", 
            "Qwen/Qwen3-VL-8B-Instruct",

            "llava-hf/llava-1.5-7b-hf", 
            "llava-hf/llava-v1.6-vicuna-7b-hf", 
            "llava-hf/llava-v1.6-mistral-7b-hf", 

            "Qwen/Qwen3-8B-Base", 
            "Qwen/Qwen3-4B-Base", 
            "Qwen/Qwen3-1.7B-Base" , 
            "Qwen/Qwen3-0.6B-Base" # doesnt work 
]

def find_matching(f, targets): 
    for t in targets : 
        if t in f : 
            f = f.replace(f'{t}', '')   
            return t, f 
    print(f"cannot find matching {f} in {targets}") 

def get_summary(dataset='mmstar'): 
    # df = df.groupby(['model', 'folder', 'condition'])['correct'].mean()

    files = glob(f"{root_dir}/*/*{dataset}*.jsonl")  + glob(f"{root_dir}/*/*/*/{dataset}*.jsonl")

    dfs= []
    for f in files: 
        try: 
            df = pd.read_json(f, lines=True)
            df['folder'] = f.split('/')[-2]
            df['model'], f = find_matching(f, [model.split('/')[-1] for model in model_names])  
            df['condition'] = f.split('/')[-1][:-6].replace(f'_', ' ').replace(f'{dataset}', '').strip()
            dfs.append(df)
        except Exception as e: 
            print(e)
    df = pd.concat(dfs)
    print(len(files) ) 

    pt = df.pivot_table(
        index=['model', 'folder'], 
        columns=['condition'], 
        values=['correct'],
        aggfunc=['mean', 'count']
    )
    # Round the "mean" rows/columns to 2 decimal places
    pt = pt.round(4)
    pt.to_csv(f"./summary_{dataset}.csv")
    return df , pt 

In [25]:
!python /home/work/yuna/HPA/evaluation/process_raw_human_responses.py


📊 Processing Raw Human Responses
   Session: s1
   Data dir: /home/work/yuna/HPA/data/humans/all_results_20251206_154732

📚 Loading annotations...

Processing VQA (text) responses...
📋 ANSWER PREPROCESSING PIPELINE

[1/5] Loading data...
✓ Loaded 641 questions from /home/work/yuna/HPA/dataset/questions/s1.csv
/home/work/yuna/HPA/data/humans/all_results_20251206_154732/1ed1a464_20251204_110002/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/all_results_20251206_154732/99f271ae_20251203_111155/answers.csv is incomplete, skip
✓ Loaded 9622 responses from 15 files

[2/5] Translating Korean answers...
✓ Loaded 1381 cached translations from /home/work/yuna/HPA/preprocessing/translation_cache.json
❌ OpenAI package not installed. Run: pip install openai
⚠ Skipping translation (no API client)

[3/5] Normalizing answers...
   ✓ /home/work/yuna/HPA/evaluation/scored/humans/cleaned_n15_choice.json
   ✓ /home/work/yuna/HPA/evaluation/scored/humans/cleaned_n15_text.json

[5/5] Savin

In [71]:
!python /home/work/yuna/HPA/evaluation/score_results.py  --input_dir finetuned  
!python /home/work/yuna/HPA/evaluation/score_results.py  --input_dir pretrained  

11424.00s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Found 5 files to score
📊 Scoring: /home/work/yuna/HPA/evaluation/results/finetuned/Qwen3-VL-4B-Instruct/A1_JS_vqa_n15_blind_inst/mmstar.jsonl
   ✓ Saved scored file: /home/work/yuna/HPA/evaluation/scored/finetuned/Qwen3-VL-4B-Instruct/A1_JS_vqa_n15_blind_inst/mmstar.jsonl

📈 Results:
   Accuracy: 0.5860 (879/1500)

   Per-category:
      coarse perception: 0.704 (176/250)
      instance reasoning: 0.700 (175/250)
      logical reasoning: 0.596 (149/250)
      math: 0.564 (141/250)
      fine-grained perception: 0.552 (138/250)
      science & technology: 0.400 (100/250)
📊 Scoring: /home/work/yuna/HPA/evaluation/results/finetuned/Qwen3-VL-4B-Instruct/A1_JS_vqa_n15_blind_inst/mmstar_inst_blind.jsonl
   ✓ Saved scored file: /home/work/yuna/HPA/evaluation/scored/finetuned/Qwen3-VL-4B-Instruct/A1_JS_vqa_n15_blind_inst/mmstar_inst_blind.jsonl

📈 Results:
   Accuracy: 0.2673 (401/1500)

   Per-category:
      math: 0.380 (95/250)
      coarse perception: 0.280 (70/250)
      fine-grained perc

In [76]:
model_results = {}
for ds in ['mmstar', 'spubench', 'vqa_5k', 'vqa_1k']: 
    model_results[ds], pv = get_summary(ds)

cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-0.6B_mmstar.jsonl in ['InternVL3_5-8B', 'InternVL3_5-2B', 'InternVL3_5-4B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-1.5-7b-hf', 'llava-v1.6-vicuna-7b-hf', 'llava-v1.6-mistral-7b-hf', 'Qwen3-8B-Base', 'Qwen3-4B-Base', 'Qwen3-1.7B-Base', 'Qwen3-0.6B-Base']
cannot unpack non-iterable NoneType object
45
cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-4B_spubench.jsonl in ['InternVL3_5-8B', 'InternVL3_5-2B', 'InternVL3_5-4B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-1.5-7b-hf', 'llava-v1.6-vicuna-7b-hf', 'llava-v1.6-mistral-7b-hf', 'Qwen3-8B-Base', 'Qwen3-4B-Base', 'Qwen3-1.7B-Base', 'Qwen3-0.6B-Base']
cannot unpack non-iterable NoneType object
40
cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-4B_vqa_5k.jsonl in ['InternVL3_5-8B', 'InternVL3_5-2B', 

In [111]:
human_mc=pd.read_csv('/home/work/yuna/HPA/evaluation/scored/humans/human_mc_per_question.csv')
human_mc['model'] = "humans" 
human_mc['folder'] = "humans" 
human_mc.rename(columns={'mean_accuracy': 'correct'}, inplace=True) 
human_mc['condition'] = "inst blind" 
human_mc.groupby('category')['correct'].mean()

category
coarse perception          0.279781
fine-grained perception    0.266330
instance reasoning         0.284211
logical reasoning          0.270775
Name: correct, dtype: float64

In [116]:
# matching pids for human-model comparison 
qids = human_mc.pid.unique()
model_mc = model_results['mmstar']
print(len(qids))
mmstar_human_comparison = pd.concat([model_mc[model_mc['pid'].isin(qids)] , human_mc])
pt = mmstar_human_comparison.pivot_table(
    index=['model', 'folder'], 
    columns=['condition'], 
    values=['correct'],
    aggfunc=['mean', 'count']
)
# Round the "mean" rows/columns to 2 decimal places
pt = pt.round(4)
pt.to_csv(f'./mmstar_human_comparison.csv')
pt 

223


mean  \
                                                                    correct   
condition                                                                     
model                    folder                                               
InternVL3_5-1B           pretrained                                0.434978   
InternVL3_5-2B           pretrained                                0.511211   
InternVL3_5-4B           pretrained                                 0.61435   
InternVL3_5-8B           pretrained                                0.609865   
Qwen3-8B-Base            pretrained                                0.197309   
Qwen3-VL-2B-Instruct     pretrained                                0.560538   
Qwen3-VL-4B-Instruct     A0_SFT_vqa_gt                             0.605381   
                         A1_JS_vqa_blind_n10                       0.596413   
                         A1_JS_vqa_n15_blind_inst                   0.58296   
                         D1_JS_mmstar_blind                        0.591928   
                         pretrained                                0.632287   
Qwen3-VL-8B-Instruct     A0_JS_vqa_gt_10                                NaN   
                         A0_SFT_vqa_gt                             0.672646   
                         A1_JS_vqa_blind_n10                       0.663677   
                         D0_SFT_mmstar_blind                       0.641256   
                         Qwen3-VL-8B-Instruct_A1_JS_vqa_blind_n15  0.654709   
                         pretrained                                0.672646   
humans                   humans                                         NaN   
llava-v1.6-mistral-7b-hf pretrained                                0.394619   

                                                                             \
                                                                              
condition                                                             blind   
model                    folder                                               
InternVL3_5-1B           pretrained                                0.295964   
InternVL3_5-2B           pretrained                                 0.26009   
InternVL3_5-4B           pretrained                                0.269058   
InternVL3_5-8B           pretrained                                0.331839   
Qwen3-8B-Base            pretrained                                     NaN   
Qwen3-VL-2B-Instruct     pretrained                                0.219731   
Qwen3-VL-4B-Instruct     A0_SFT_vqa_gt                                  NaN   
                         A1_JS_vqa_blind_n10                       0.286996   
                         A1_JS_vqa_n15_blind_inst                       NaN   
                         D1_JS_mmstar_blind                         0.29148   
                         pretrained                                0.313901   
Qwen3-VL-8B-Instruct     A0_JS_vqa_gt_10                                NaN   
                         A0_SFT_vqa_gt                                  NaN   
                         A1_JS_vqa_blind_n10                            NaN   
                         D0_SFT_mmstar_blind                            NaN   
                         Qwen3-VL-8B-Instruct_A1_JS_vqa_blind_n15       NaN   
                         pretrained                                0.295964   
humans                   humans                                         NaN   
llava-v1.6-mistral-7b-hf pretrained                                     NaN   

                                                                              \
                                                                               
condition                                                         inst blind   
model                    folder                                                
InternVL3_5-1B           pretrained                                 0.282511   
InternVL3_5-2B           pretrained

In [67]:
human_vqa=pd.read_csv('/home/work/yuna/HPA/evaluation/scored/humans/human_vqa_per_question.csv')
# matching pids for human-model comparison 
human_vqa['model'] = "humans" 

qids = human_vqa.qid.unique()
print(len(qids))
model_vqa = model_results['vqa_1k'] 
model_vqa[model_vqa['question_id'].isin(qids)] 

374

In [ ]:
pt = df.pivot_table(
    index=['model', 'folder'], 
    columns=['condition'], 
    values=['correct'],
    aggfunc=['mean', 'count']
)
# Round the "mean" rows/columns to 2 decimal places
pt = pt.round(4)

# MMStar 

In [44]:
dfs = []
for filepath in glob("/home/work/yuna/HPA/results/swift/*mmstar*.jsonl"): 
    print(f"Evaluating: {filepath}")
    df = read_file(filepath) 
    dfs.append(df)
    # results = evaluate_results(filepath)
    # print_report(results)

Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-0.6B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v

In [54]:
df = pd.concat(dfs)
df = df[df['condition'] != '_blind']
results = df.groupby(['model_full', 'condition', 'category', 'l2_category'])['correct'].mean().reset_index()
results.pivot_table(index=['model_full', 'condition'], columns=[ 'category', 'l2_category'], values=['correct'])

correct  \
category                                      coarse perception   
l2_category                                       image emotion   
model_full                        condition                       
OpenGVLab/InternVL3_5-2B          _inst_blind          0.193548   
OpenGVLab/InternVL3_5-4B                               0.000000   
                                  _inst_blind          0.258065   
OpenGVLab/InternVL3_5-8B                               0.774194   
                                  _inst_blind          0.451613   
Qwen/Qwen3-VL-4B-Instruct                              0.161290   
                                  _inst_blind          0.322581   
Qwen/Qwen3-VL-8B-Instruct                              0.483871   
                                  _inst_blind          0.161290   
llava-hf/llava-v1.6-mistral-7b-hf                      0.580645   
                                  _inst_blind          0.032258   

                                                                     \
category                                                              
l2_category                                   image scene and topic   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.042553   
OpenGVLab/InternVL3_5-4B                                   0.047619   
                                  _inst_blind              0.063830   
OpenGVLab/InternVL3_5-8B                                   0.588652   
                                  _inst_blind              0.283688   
Qwen/Qwen3-VL-4B-Instruct                                  0.411348   
                                  _inst_blind              0.148936   
Qwen/Qwen3-VL-8B-Instruct                                  0.425532   
                                  _inst_blind              0.198582   
llava-hf/llava-v1.6-mistral-7b-hf                          0.453901   
                                  _inst_blind              0.212766   

                                                                     \
category                                                              
l2_category                                   image style & quality   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.038462   
OpenGVLab/InternVL3_5-4B                                   0.333333   
                                  _inst_blind              0.000000   
OpenGVLab/InternVL3_5-8B                                   0.769231   
                                  _inst_blind              0.333333   
Qwen/Qwen3-VL-4B-Instruct                                  0.397436   
                                  _inst_blind              0.153846   
Qwen/Qwen3-VL-8B-Instruct                                  0.666667   
                                  _inst_blind              0.141026   
llava-hf/llava-v1.6-mistral-7b-hf                          0.628205   
                                  _inst_blind              0.089744   

                                                                       \
category                                      fine-grained perception   
l2_category                                              localization   
model_full                        condition                             
OpenGVLab/InternVL3_5-2B          _inst_blind                   0.000   
OpenGVLab/InternVL3_5-4B                                          NaN   
                                  _inst_blind                   0.100   
OpenGVLab/InternVL3_5-8B                                        0.675   
                                  _inst_blind                   0.250   
Qwen/Qwen3-VL-4B-Instruct                                       0.325   
                                  _inst_blind                   0.150   
Qwen/Qwen3-VL-8B-Instruct                                       0.550   
                                  _inst_bl